# Results Analysis: Predictive Sales

## Goal
Analyze model performance, compare results across different approaches, and identify areas for improvement.

## Contents
1. Model Results Comparison
2. Tuned Model Evaluation
3. Feature Importance Analysis
4. Misclassification Analysis
5. Threshold Optimization
6. Feature Engineering Improvements

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from xgboost import XGBClassifier

# Set style
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load Data
try:
    df = pd.read_csv("../data/processed/processed_sales_data.csv")
except FileNotFoundError:
    df = pd.read_csv("data/processed/processed_sales_data.csv")

print(f"Loaded data shape: {df.shape}")

## 1. Data Preparation and Model Setup

In [ ]:
# Define features (excluding leaky features)
features = [
    'sector', 'year_established', 'revenue', 'employees', 'office_location', 
    'series', 'sales_price', 'product', 'manager', 'regional_office', 
    'engage_year', 'engage_month', 'log_revenue', 'log_employees'
]

target = 'target'

# Drop missing values
df = df.dropna(subset=features)

X = df[features]
y = df[target]

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Define preprocessor
categorical_features = ['sector', 'office_location', 'series', 'product', 'manager', 'regional_office']
numeric_features = ['year_established', 'revenue', 'employees', 'sales_price', 'engage_year', 'engage_month', 'log_revenue', 'log_employees']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")

## 2. Model Results Comparison
Compare ROC-AUC scores across different model approaches.

In [ ]:
# Train all models and collect results
results = {}

# Logistic Regression
lr_model = Pipeline([('preprocessing', preprocessor), 
                     ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
lr_model.fit(X_train, y_train)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]
results['Logistic Regression'] = roc_auc_score(y_test, y_prob_lr)

# Random Forest
rf_model = Pipeline([('preprocessing', preprocessor), 
                     ('clf', RandomForestClassifier(n_estimators=200, random_state=42))])
rf_model.fit(X_train, y_train)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
results['Random Forest'] = roc_auc_score(y_test, y_prob_rf)

# XGBoost
xgb_model = Pipeline([('preprocessing', preprocessor), 
                      ('clf', XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, 
                                           subsample=0.8, colsample_bytree=0.8, 
                                           eval_metric='logloss', random_state=42))])
xgb_model.fit(X_train, y_train)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
results['XGBoost'] = roc_auc_score(y_test, y_prob_xgb)

# Display results
print("MODEL COMPARISON (ROC-AUC Scores):")
for model_name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {model_name}: {score:.4f}")

## 3. Tuned XGBoost Model Evaluation
Evaluate the best XGBoost model from hyperparameter tuning.

In [ ]:
# Use best parameters from grid search (or default tuned)
best_xgb = Pipeline([
    ('preprocessing', preprocessor),
    ('clf', XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42
    ))
])

best_xgb.fit(X_train, y_train)

y_pred_best = best_xgb.predict(X_test)
y_prob_best = best_xgb.predict_proba(X_test)[:, 1]

print("TUNED XGBOOST RESULTS")
print("F1 Score:", f1_score(y_test, y_pred_best))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_best))
print(classification_report(y_test, y_pred_best))

## 4. Feature Importance Analysis
Identify the most influential features for predictions.

In [ ]:
# Get feature names after preprocessing
feature_names = best_xgb.named_steps['preprocessing'].get_feature_names_out()

importances = best_xgb.named_steps['clf'].feature_importances_

# Sort top 15
indices = np.argsort(importances)[-15:]

plt.figure(figsize=(10, 8))
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), feature_names[indices])
plt.title("Top 15 Feature Importances (XGBoost)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 5. Misclassification Analysis
Analyze which samples the model misclassified.

In [ ]:
misclassified = X_test.copy()
misclassified['true'] = y_test.values
misclassified['pred'] = y_pred_best

errors = misclassified[misclassified['true'] != misclassified['pred']]

print("Number of misclassified samples:", len(errors))
print(f"Error rate: {len(errors)/len(X_test):.2%}")
print("\nMisclassified samples preview:")
errors.head()

In [ ]:
# Analyze misclassifications by sector
print("Misclassifications by Sector:")
print(errors['sector'].value_counts())

## 6. Threshold Optimization
Find the optimal probability threshold for classification.

In [ ]:
probs = y_prob_best

thresholds = np.linspace(0.1, 0.9, 50)
f1_scores = []

for t in thresholds:
    preds = (probs >= t).astype(int)
    f1_scores.append(f1_score(y_test, preds))

best_threshold = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

print("Best Threshold:", best_threshold)
print("Best F1 Score:", best_f1)

# Plot threshold vs F1
plt.figure(figsize=(10, 5))
plt.plot(thresholds, f1_scores, marker='o', markersize=3)
plt.axvline(best_threshold, color='red', linestyle='--', label=f'Best threshold: {best_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('F1 Score vs Classification Threshold')
plt.legend()
plt.grid(True)
plt.show()

## 7. Feature Engineering Improvements
Add new engineered features and evaluate impact.

In [ ]:
# Reload data for feature engineering
df_enhanced = pd.read_csv("../data/processed/processed_sales_data.csv")

# Company age (proxy for maturity)
df_enhanced['company_age'] = df_enhanced['engage_year'] - df_enhanced['year_established']

# Revenue per employee
df_enhanced['revenue_per_employee'] = df_enhanced['revenue'] / df_enhanced['employees']

# Interaction features
df_enhanced['duration_x_revenue'] = df_enhanced['deal_duration_days'] * df_enhanced['log_revenue']
df_enhanced['duration_x_employees'] = df_enhanced['deal_duration_days'] * df_enhanced['log_employees']

print("New features added:")
print(df_enhanced[['company_age', 'revenue_per_employee', 'duration_x_revenue', 'duration_x_employees']].head())

In [ ]:
# Prepare updated features
X_updated = df_enhanced.drop(columns=['target', 'deal_stage'])
y_updated = df_enhanced['target']

# Get updated column types
categorical_cols_u = X_updated.select_dtypes(include='object').columns.tolist()
numeric_cols_u = X_updated.select_dtypes(exclude='object').columns.tolist()

X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    X_updated, y_updated,
    test_size=0.2,
    random_state=42,
    stratify=y_updated
)

# Updated preprocessor
preprocessor_updated = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols_u),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols_u)
    ]
)

# Train updated model
xgb_updated = Pipeline([
    ('preprocessing', preprocessor_updated),
    ('clf', XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42
    ))
])

xgb_updated.fit(X_train_u, y_train_u)

y_pred_u = xgb_updated.predict(X_test_u)
y_prob_u = xgb_updated.predict_proba(X_test_u)[:, 1]

print("UPDATED XGBOOST RESULTS (with new features)")
print("F1 Score:", f1_score(y_test_u, y_pred_u))
print("ROC-AUC:", roc_auc_score(y_test_u, y_prob_u))
print(classification_report(y_test_u, y_pred_u))

## 8. Summary
Key findings and recommendations based on the analysis.

In [ ]:
print("=" * 50)
print("RESULTS SUMMARY")
print("=" * 50)
print("\n1. MODEL COMPARISON:")
for model_name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"   - {model_name}: ROC-AUC = {score:.4f}")

print(f"\n2. BEST THRESHOLD: {best_threshold:.2f} (F1 = {best_f1:.4f})")

print(f"\n3. MISCLASSIFICATION RATE: {len(errors)/len(X_test):.2%}")

print("\n4. KEY RECOMMENDATIONS:")
print("   - XGBoost with tuned hyperparameters provides best performance")
print("   - Feature engineering (company_age, revenue_per_employee) can improve results")
print("   - Adjusting classification threshold can optimize for specific business needs")
print("   - Focus on sectors with higher misclassification rates for improvement")